# Vision Access — export the 60-class YOLOE model

This re-exports `public/models/yoloe-11s.onnx` for the expanded 60-class
vocabulary in `model/classes.txt`. It needs a GPU and downloads MobileCLIP
(~570 MB) plus the YOLOE checkpoint, which is why this runs in Colab rather
than on a laptop.

**Before you start:** Runtime -> Change runtime type -> Hardware accelerator
-> **GPU** (a free T4 is fine), then Save.

Steps below: check the GPU, upload three small files from your repo's
`model/` folder, install dependencies, run the export, download the result.

In [ ]:
!nvidia-smi

## 1. Upload the export script and the class list

Run the cell below, then in the file picker select **all three** of these
files from your local `model/` folder:

- `yoloe_export.py`
- `requirements.txt`
- `classes.txt` (regenerate it first with `npx tsx scripts/export-classes.ts`
  if you've touched `lib/phrases.ts` since the last export — it should say
  `# 60 classes.` at the top)

They need to land inside a `model/` folder, matching the repo layout, because
the script resolves paths relative to its own location.

In [ ]:
import os
from google.colab import files

os.makedirs("/content/vision-access/model", exist_ok=True)
os.chdir("/content/vision-access/model")

uploaded = files.upload()
expected = {"yoloe_export.py", "requirements.txt", "classes.txt"}
missing = expected - set(uploaded.keys())
if missing:
    raise SystemExit(f"Missing: {missing} — re-run this cell and select all three.")
print("OK:", sorted(uploaded.keys()))

In [ ]:
!head -5 classes.txt
!echo ...
!wc -l classes.txt

Confirm the header above says **60 classes**. If it says 32, you uploaded the
old `classes.txt` — regenerate it locally and re-upload before continuing.

## 2. Install dependencies

Pulls `ultralytics` (YOLOE support), `onnx`, `onnxslim`, `onnxruntime`, and
PyTorch as a transitive dependency (~2.5 GB, several minutes).

In [ ]:
!pip install -q -r requirements.txt

## 3. Export

`--no-run` skips the live-inference sanity loop (there is no webcam in Colab)
and goes straight to setting the 60-class vocabulary and exporting. This is
the expensive step: it downloads the YOLOE checkpoint and runs MobileCLIP's
text encoder once over all 60 prompts, then exports, strips the segmentation
outputs, and verifies the strip against an unstripped reference — all
automatic, per `model/yoloe_export.py`.

In [ ]:
!python3 yoloe_export.py --classes classes.txt --no-run --export

Check the printed `INSPECT` section near the end: it must say
**`OK: output0 has 64 channels = 4 + 60 classes.`** (4 box channels + 60). If
it instead reports a SEGMENTATION EXPORT or a channel-count warning, stop —
do not download the file — and re-run the export cell; something upstream
(usually a stale `ultralytics` cache) needs clearing first.

## 4. (Optional) sanity-check on a real photo

Skip this if you're confident. Otherwise upload one or two photos taken from
roughly chest height on a UB street/sidewalk and see which of the 60 classes
actually fire — the same per-class hit table `model/yoloe_export.py` prints
when run normally (without `--no-run`). A class that never fires on
representative photos is a class you'll never hear in the app no matter how
good its `.wav` file is, and is a candidate to cut before the next export.

In [ ]:
from google.colab import files
os.makedirs("samples", exist_ok=True)
os.chdir("samples")
files.upload()
os.chdir("..")

In [ ]:
!python3 yoloe_export.py --classes classes.txt --source samples/ --frames 0

## 5. Download the exported model

This grabs `yoloe-11s.onnx` from `model/`. If a previous cell in this notebook
printed a different filename (checkpoint naming can vary), adjust the path
below to match before running.

In [ ]:
from google.colab import files
files.download("yoloe-11s.onnx")

## 6. Back on your machine

1. Replace `public/models/yoloe-11s.onnx` with the downloaded file.
2. In `lib/vision.worker.ts`, bump `MODEL_CACHE` (e.g. `"vision-model-v3"` ->
   `"vision-model-v4"`) — without this, a phone that already cached the old
   32-class model in Cache Storage keeps serving those bytes indefinitely.
3. Run the app locally and confirm the on-screen debug readout (fps / model
   status line) shows detections with names again, not just distances.
4. Once you've confirmed it works, `LEGACY_32_CLASSES` in
   `lib/vision.worker.ts` becomes dead weight you can remove — it only exists
   to keep the OLD 32-class model naming things correctly while this new one
   was being built.